In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import pandas as pd
import keras_nlp
import keras_core as keras
from sklearn.model_selection import train_test_split

# バックエンドを TensorFlow に設定
os.environ["KERAS_BACKEND"] = "tensorflow"

# データ読み込み
df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df_test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

In [ ]:
df_train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_train.head(25)
##テスト用ファイル
##災害ではないのが０
##災害が１

In [ ]:
df_test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
df_test.head()
#print(len(df))
##本番用ファイル

In [ ]:
df_sample = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
df_sample.head()
##提出用ファイル

In [ ]:
# テキストの表示幅を広げる（省略されないようにする）
df_train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_train.head()
##テスト用ファイル

pd.set_option('display.max_colwidth', None)
df_train.head()

In [ ]:
df_train.info()##データの情報
df_train.describe()##データの統計
df_train.isnull().sum()##何件欠損があるか

In [ ]:
print(len(df_train))
df_train['target'].value_counts()
##災害ではないのが０
##災害が１

In [ ]:
df_train.head(30)

In [ ]:
df_train.head(25)

In [ ]:
# 欠損を埋める（空文字）
df_train["text"] = df_train["text"].fillna("")
df_test["text"] = df_test["text"].fillna("")

# データ分割
X_train, X_val, y_train, y_val = train_test_split(df_train["text"], df_train["target"], test_size=0.2, random_state=42)

In [ ]:
classifier = keras_nlp.models.BertClassifier.from_preset(
    "bert_base_en_uncased",
    num_classes=2,
    activation="softmax"
)

import tensorflow as tf  

classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),  
    metrics=["accuracy"]
)

In [ ]:
classifier.fit(
    X_train.to_list(), y_train,
    validation_data=(X_val.to_list(), y_val),
    epochs=2,
    batch_size=16
)

In [ ]:
# 📤 テスト用の予測
test_probs = classifier.predict(df_test["text"].to_list(), batch_size=16)
test_preds = test_probs.argmax(axis=1)  # softmaxの最大値 → クラス番号

# 📝 提出ファイルの作成
submission = pd.DataFrame({
    "id": df_test["id"],
    "target": test_preds
})

submission.to_csv("submission2.csv", index=False)